# GEO · E1.5 — teaching the RULE (agi-semantic-core Phase 8, flight 2)

**The question:** E1 proved the dictionary's angular contract can be instilled as a *map* (held-out pairs of seen concepts land at their true angles) but not as a *grammar* (unseen antonym words did not decompress — the 17 twins went 57.0° → 52.5°). This flight adds an **external antonymy channel** — WordNet antonym pairs trained to 90° — and asks whether "opposition → orthogonal" then generalizes to words the model never saw in training.

**Design:**
- **V2ref** — the E1 V2 recipe retrained under identical conditions (internal reference)
- **W0_rule_only** — WordNet antonym+synonym channels + retention, NO dictionary (isolates the rule)
- **W1_full** — V2 recipe + the rule channels (the production candidate)
- WordNet pairs where **both** words are dictionary vocabulary are excluded from the rule channels — the dictionary stays sovereign over its own concepts. The 40 twin-probe words are excluded from all training. Antonym evaluation is **word-disjoint**: held-out pairs share no words with training pairs, reported in *any-held* / *both-held* strata and split *morphological vs lexical* (lexical both-held is the strictest rule test).
- Pre-registered verdict criteria are printed before judgment.

**How to run:** Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save · Runtime ▸ **Run all**. ≈60–90 min, free tier. Re-running resumes past trained variants.

**Output:** `geo_e15_results.zip` (metrics, verdict, per-pair tables, best model) — browser download at the end, optional Drive cell last.

*Staged 2026-08-20 · follows E0/E1 (results: pass doc §8) and E3 (§9) · design: `~/qualia-algebra/internal/GEOMETRIC-REASONING-PASS.md`*

In [ ]:
# ── Setup: GPU check, installs, repo clone, WordNet ──────────────────────────
import subprocess, sys
from pathlib import Path

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')
import torch
assert torch.cuda.is_available(), (
    'No GPU. Colab menu: Runtime > Change runtime type > Hardware accelerator: T4 GPU, then Run all again.')

print('Installing packages (~1-2 min)...')
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'sentence-transformers>=3.0','transformers','accelerate','datasets',
    'scikit-learn','scipy','pandas','nltk'], check=True)
import nltk
nltk.download('wordnet')   # loud on purpose; cell 3 self-heals if this fails

REPO = Path('/content/agi-semantic-core')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/QAv2/agi-semantic-core.git', str(REPO)], check=True)
DB = REPO/'db'/'semantic.db'
assert DB.exists(), 'semantic.db missing from clone'

RESULTS = Path('/content/geo15_results'); RESULTS.mkdir(exist_ok=True)
import sqlite3
n_concepts = sqlite3.connect(DB).execute('SELECT COUNT(*) FROM concepts').fetchone()[0]
print(f'Dictionary loaded: {n_concepts} concepts (expected 3033)')
SEED = 42

In [ ]:
# ── Dictionary extraction — IDENTICAL to E0/E1 (same seed → same splits) ─────
import numpy as np, sqlite3
from collections import Counter
rng = np.random.default_rng(SEED)

con = sqlite3.connect(DB); con.row_factory = sqlite3.Row
DIMS = ['x','y','z','e','f','g','h','fx','fy','fz','fe','ff','fg','fh']
rows = con.execute(f"SELECT id,name,description,{','.join(DIMS)} FROM concepts ORDER BY id").fetchall()
names = [r['name'] for r in rows]
texts = [f"{r['name']}: {r['description']}" for r in rows]
Y14 = np.array([[r[d] for d in DIMS] for r in rows], dtype=np.float64)
Y7 = Y14[:, :7]
id2idx = {r['id']: i for i, r in enumerate(rows)}
name_set  = {r['name'].upper() for r in rows}
alias_set = {a['alias'].upper() for a in con.execute('SELECT alias FROM aliases')}

pairs = []
for r in con.execute("SELECT concept1_id c1, concept2_id c2, rel_type, angle_4d, angle_8d FROM relations"):
    if r['c1'] not in id2idx or r['c2'] not in id2idx: continue
    ta = r['angle_8d'] if r['angle_8d'] else r['angle_4d']
    if not ta: continue
    pairs.append((id2idx[r['c1']], id2idx[r['c2']], r['rel_type'], float(ta)))
print(f'{len(pairs)} usable relations —', dict(Counter(p[2] for p in pairs)))

by_type = {}
for p in pairs: by_type.setdefault(p[2], []).append(p)
train_rel, test_rel = [], []
for t, ps in sorted(by_type.items()):
    idx = rng.permutation(len(ps)); cut = int(0.8*len(ps))
    train_rel += [ps[i] for i in idx[:cut]]; test_rel += [ps[i] for i in idx[cut:]]
print(f'relation split: {len(train_rel)} train / {len(test_rel)} held out (same as E1)')

def angle7(i, j):
    a, b = Y7[i], Y7[j]
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-9 or nb < 1e-9: return None
    return float(np.degrees(np.arccos(np.clip(a@b/(na*nb), -1, 1))))
related = {(min(p[0],p[1]), max(p[0],p[1])) for p in pairs}
rand_pairs, seen = [], set()
while len(rand_pairs) < 12000:
    i, j = (int(v) for v in rng.integers(0, len(names), 2))
    key = (min(i,j), max(i,j))
    if i == j or key in related or key in seen: continue
    a = angle7(i, j)
    if a is None: continue
    seen.add(key); rand_pairs.append((i, j, 'random', a))
rand_train, rand_test = rand_pairs[:10000], rand_pairs[10000:]
print(f'random-pair channel: {len(rand_train)} train / {len(rand_test)} held out')

In [ ]:
# ── WordNet rule channels: antonyms→90°, synonyms→tight; word-disjoint splits ─
import numpy as np, nltk, os, urllib.request, zipfile

def ensure_wordnet():
    try:
        nltk.data.find('corpora/wordnet'); return
    except LookupError:
        pass
    nltk.download('wordnet')                       # retry, loud
    nltk.download('omw-1.4')                       # companion, best-effort
    try:
        nltk.data.find('corpora/wordnet'); return
    except LookupError:
        pass
    root = os.path.expanduser('~/nltk_data/corpora')   # manual mirror fallback
    os.makedirs(root, exist_ok=True)
    zp = os.path.join(root, 'wordnet.zip')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/wordnet.zip', zp)
    with zipfile.ZipFile(zp) as z: z.extractall(root)
    nltk.data.find('corpora/wordnet')              # raises loudly if still broken
ensure_wordnet()
from nltk.corpus import wordnet as wn
rng_wn = np.random.default_rng(SEED + 1)

TWINS = [('damp','arid'),('sprint','stroll'),('whisper','shout'),('ascend','plummet'),
 ('inflate','deflate'),('gather','scatter'),('freeze','thaw'),('arrive','depart'),
 ('absorb','emit'),('expand','contract'),('attack','defend'),('borrow','lend'),
 ('buy','sell'),('float','sink'),('melt','solidify'),('sharpen','dull'),
 ('tighten','loosen'),('accelerate','decelerate'),('brighten','darken'),
 ('strengthen','weaken'),('appear','vanish'),('assemble','disassemble'),
 ('encourage','discourage'),('inhale','exhale'),('import','export'),
 ('maximize','minimize'),('ancient','futuristic'),('crowded','deserted'),
 ('fertile','barren'),('flexible','rigid'),('generous','stingy'),('humble','arrogant'),
 ('innocent','guilty'),('optimist','pessimist'),('permanent','temporary'),
 ('scarce','abundant'),('shallow','profound'),('smooth','jagged'),('tame','feral'),
 ('transparent','opaque')]
twin_words = {w for p in TWINS for w in p}
dict_vocab = name_set | alias_set

ant = set()
for s in wn.all_synsets():
    for l in s.lemmas():
        for a in l.antonyms():
            w1, w2 = l.name().lower(), a.name().lower()
            if w1.isalpha() and w2.isalpha() and w1 != w2 and len(w1) > 2 and len(w2) > 2:
                ant.add((min(w1, w2), max(w1, w2)))
syn_pairs = set()
for s in wn.all_synsets():
    lem = sorted({l.name().lower() for l in s.lemmas() if l.name().isalpha() and len(l.name()) > 2})
    for i in range(len(lem)):
        for j in range(i+1, len(lem)):
            syn_pairs.add((lem[i], lem[j]))
syn_pairs -= ant
print(f'WordNet raw: {len(ant)} antonym pairs, {len(syn_pairs)} synonym pairs')

def rule_eligible(p):
    a, b = p
    if a in twin_words or b in twin_words: return False               # twins stay untouched
    if a.upper() in dict_vocab and b.upper() in dict_vocab: return False  # dictionary is sovereign
    return True
ant_e  = sorted(p for p in ant if rule_eligible(p))
syn_e  = sorted(p for p in syn_pairs if rule_eligible(p))
print(f'eligible after exclusions: {len(ant_e)} antonym, {len(syn_e)} synonym')

# word-disjoint antonym split: hold out 20% of words; train pairs touch NO held word
ant_vocab = sorted({w for p in ant_e for w in p})
held_w = set(np.array(ant_vocab)[rng_wn.permutation(len(ant_vocab))[:int(0.2*len(ant_vocab))]])
ant_train = [p for p in ant_e if p[0] not in held_w and p[1] not in held_w]
ant_any   = [p for p in ant_e if (p[0] in held_w) != (p[1] in held_w)]
ant_both  = [p for p in ant_e if p[0] in held_w and p[1] in held_w]
def is_morph(p): return (p[0] in p[1]) or (p[1] in p[0]) or (min(len(p[0]),len(p[1]))>=5 and p[0][:4]==p[1][:4])
ant_both_lex   = [p for p in ant_both if not is_morph(p)]
ant_both_morph = [p for p in ant_both if is_morph(p)]
print(f'antonym split: train {len(ant_train)} / any-held {len(ant_any)} / '
      f'both-held {len(ant_both)} (lexical {len(ant_both_lex)}, morphological {len(ant_both_morph)})')

# word-disjoint synonym split: hold out 10% of words
syn_vocab = sorted({w for p in syn_e for w in p})
held_sw = set(np.array(syn_vocab)[rng_wn.permutation(len(syn_vocab))[:int(0.1*len(syn_vocab))]])
syn_pool  = [p for p in syn_e if p[0] not in held_sw and p[1] not in held_sw]
syn_train = [syn_pool[i] for i in rng_wn.permutation(len(syn_pool))[:8000]]
syn_both  = [p for p in syn_e if p[0] in held_sw and p[1] in held_sw]
syn_eval  = [syn_both[i] for i in rng_wn.permutation(len(syn_both))[:800]]
print(f'synonym: train {len(syn_train)} / word-disjoint eval {len(syn_eval)}')

# random-word control (neither antonym nor synonym) — selectivity check, eval-only
wn_vocab = sorted(set(ant_vocab) | set(syn_vocab))
antsyn = ant | syn_pairs
ctrl = []
while len(ctrl) < 1500:
    i, j = rng_wn.integers(0, len(wn_vocab), 2)
    a, b = wn_vocab[int(i)], wn_vocab[int(j)]
    if a == b: continue
    key = (min(a,b), max(a,b))
    if key in antsyn: continue
    ctrl.append(key)
print(f'random-word control pairs: {len(ctrl)}')

In [ ]:
# ── Datasets: V2ref / W0_rule_only / W1_full + frozen-teacher distillation ────
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

def cosd(deg): return float(np.cos(np.radians(deg)))
def pair_rows(plist, reps=1):
    return [{'sentence1': texts[i], 'sentence2': texts[j], 'score': cosd(ta)}
            for i, j, t, ta in plist] * reps
def word_rows(plist, score, reps=1):
    return [{'sentence1': a, 'sentence2': b, 'score': float(score)} for a, b in plist] * reps

stsb = load_dataset('sentence-transformers/stsb')
sts_train, sts_dev = stsb['train'], stsb['validation']
teacher = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
ta_ = teacher.encode(list(sts_train['sentence1']), batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
tb_ = teacher.encode(list(sts_train['sentence2']), batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
t_cos = (ta_ * tb_).sum(1)
distill_rows = [{'sentence1': s1, 'sentence2': s2, 'score': float(c)}
                for s1, s2, c in zip(sts_train['sentence1'], sts_train['sentence2'], t_cos)]
del ta_, tb_, teacher
import torch; torch.cuda.empty_cache()

dict_rows = pair_rows(train_rel, reps=2) + pair_rows(rand_train)
ANT_TARGET, SYN_TARGET = 0.0, 0.9
ant_rows = word_rows(ant_train, ANT_TARGET, reps=3)
syn_rows = word_rows(syn_train, SYN_TARGET)

VARIANTS = {
    'V2ref':        dict_rows + distill_rows*4,
    'W0_rule_only': ant_rows + syn_rows + distill_rows*2,
    'W1_full':      dict_rows + distill_rows*4 + ant_rows + syn_rows,
}
for k, v in VARIANTS.items(): print(f'{k}: {len(v)} examples')

In [ ]:
# ── Train the three variants (skips any already trained) ─────────────────────
import numpy as np, torch, random, shutil
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)
MODELS = RESULTS/'models'; MODELS.mkdir(exist_ok=True)
for vname, rows_v in VARIANTS.items():
    outdir = MODELS/vname
    if (outdir/'config.json').exists():
        print(vname, 'already trained — skip'); continue
    print(f'=== training {vname} ({len(rows_v)} examples) ===')
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
    ds = Dataset.from_list(rows_v).shuffle(seed=SEED)
    args = SentenceTransformerTrainingArguments(
        output_dir=f'/content/tmp_{vname}', num_train_epochs=4,
        per_device_train_batch_size=64, learning_rate=2e-5, warmup_ratio=0.1,
        fp16=True, logging_steps=400, save_strategy='no', report_to='none', seed=SEED)
    SentenceTransformerTrainer(model=model, args=args, train_dataset=ds,
                               loss=losses.CosineSimilarityLoss(model)).train()
    model.save(str(outdir))
    del model; torch.cuda.empty_cache()
    shutil.rmtree(f'/content/tmp_{vname}', ignore_errors=True)
print('training done')

In [ ]:
# ── Evaluation battery: rule generalization + map retention + semantics ───────
import numpy as np, json, torch, pandas as pd
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer

def word_angles(model, plist):
    if not plist: return np.array([])
    va = model.encode([a for a, b in plist], batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
    vb = model.encode([b for a, b in plist], batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
    return np.degrees(np.arccos(np.clip((va*vb).sum(1), -1, 1)))

def dict_pair_angles(model, plist):
    idxs = sorted({i for p in plist for i in p[:2]})
    sub = model.encode([texts[i] for i in idxs], batch_size=256,
                       convert_to_numpy=True, normalize_embeddings=True)
    pos = {ix: k for k, ix in enumerate(idxs)}
    return [(t, ta, float(np.degrees(np.arccos(np.clip(sub[pos[i]] @ sub[pos[j]], -1, 1)))))
            for i, j, t, ta in plist]

tw_keep = [(a,b) for a,b in TWINS if a.upper() not in dict_vocab and b.upper() not in dict_vocab]
print(f'twin probe: {len(tw_keep)} pairs (same set as E1)')

def eval_model(model, tag):
    r = {'tag': tag}
    for label, plist in [('ant_any', ant_any), ('ant_both', ant_both),
                         ('ant_both_lex', ant_both_lex), ('ant_both_morph', ant_both_morph),
                         ('twins', tw_keep), ('syn_held', syn_eval), ('ctrl', ctrl)]:
        ang = word_angles(model, plist)
        if len(ang):
            r[f'{label}_mean'] = float(ang.mean())
            r[f'{label}_n'] = len(ang)
            if label.startswith('ant') or label == 'twins':
                r[f'{label}_pct75'] = float(100*(ang >= 75).mean())
    ang = dict_pair_angles(model, test_rel)
    for t in ['complement', 'synonym']:
        sel = [(ta, pa) for (tt, ta, pa) in ang if tt == t]
        ta_, pa_ = map(np.array, zip(*sel))
        r[f'dict_{t}_abs_err'] = float(np.abs(pa_ - ta_).mean())
    a = model.encode(list(sts_dev['sentence1']), batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
    b = model.encode(list(sts_dev['sentence2']), batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
    r['stsb'] = float(spearmanr((a*b).sum(1), list(sts_dev['score']))[0])
    return r

results, twin_detail = [], {'pairs': [f'{a}/{b}' for a, b in tw_keep]}
mods = [('baseline', 'sentence-transformers/all-MiniLM-L6-v2')] +        [(v, str(MODELS/v)) for v in VARIANTS]
for tag, path in mods:
    m = SentenceTransformer(path, device='cuda')
    results.append(eval_model(m, tag))
    twin_detail[tag] = [float(x) for x in word_angles(m, tw_keep)]
    del m; torch.cuda.empty_cache()

df = pd.DataFrame(results).set_index('tag')
pd.set_option('display.width', 240)
cols = ['ant_both_lex_mean','ant_both_morph_mean','ant_any_mean','twins_mean',
        'syn_held_mean','ctrl_mean','dict_complement_abs_err','dict_synonym_abs_err','stsb']
print(df[cols].round(2))
df.to_csv(RESULTS/'e15_results.csv')
json.dump(twin_detail, open(RESULTS/'e15_twins.json','w'), indent=1)
json.dump({'ant_both_lex': ant_both_lex, 'ant_both_morph': ant_both_morph},
          open(RESULTS/'e15_heldout_pairs.json','w'), indent=1)

In [ ]:
# ── Pre-registered verdict ────────────────────────────────────────────────────
import json, pandas as pd
df = pd.read_csv(RESULTS/'e15_results.csv').set_index('tag')
base = df.loc['baseline']
print('criteria — RULE-SUCCESS: both-held LEXICAL antonyms mean >= 72 AND twins mean >= 65')
print('           AND STS-B drop < 0.03 AND (W1 only) dict complement err <= 12')
print('           RULE-PARTIAL: any-held antonyms mean >= 63 AND STS-B drop < 0.08; else FAIL')
print(f"baseline reference: ant_both_lex {base.ant_both_lex_mean:.1f} | twins {base.twins_mean:.1f} | "
      f"ctrl {base.ctrl_mean:.1f} | stsb {base.stsb:.4f}")
verdict = {'baseline': {c: float(base[c]) for c in df.columns}}
for v in ['V2ref', 'W0_rule_only', 'W1_full']:
    r = df.loc[v]
    drop = base.stsb - r.stsb
    ok_map = (v != 'W1_full') or (r.dict_complement_abs_err <= 12)
    if r.ant_both_lex_mean >= 72 and r.twins_mean >= 65 and drop < 0.03 and ok_map:
        status = 'RULE-SUCCESS'
    elif r.ant_any_mean >= 63 and drop < 0.08:
        status = 'RULE-PARTIAL'
    else:
        status = 'RULE-FAIL'
    verdict[v] = {'status': status, 'stsb_drop': float(drop),
                  **{c: float(r[c]) for c in df.columns}}
    print(f"{v:14s} lex-both {r.ant_both_lex_mean:5.1f}  morph-both {r.ant_both_morph_mean:5.1f}  "
          f"any {r.ant_any_mean:5.1f}  twins {r.twins_mean:5.1f}  ctrl {r.ctrl_mean:5.1f}  "
          f"dict-comp {r.dict_complement_abs_err:4.1f}  drop {drop:+.3f}  -> {status}")
json.dump(verdict, open(RESULTS/'e15_verdict.json','w'), indent=1)
print()
print('Reading guide: LEXICAL both-held is the true rule test (no morphological shortcut).')
print('ctrl_mean should stay near baseline — a large rise means indiscriminate spreading,')
print('not a learned rule. Twins vs E1 (V2 sent them DOWN to ~52°) is the headline delta.')

In [ ]:
# ── Package results (+ browser download) ──────────────────────────────────────
import shutil, json
from pathlib import Path
PKG = Path('/content/geo15_pkg'); shutil.rmtree(PKG, ignore_errors=True); PKG.mkdir()
for f in ['e15_results.csv','e15_twins.json','e15_verdict.json','e15_heldout_pairs.json']:
    p = RESULTS/f
    if p.exists(): shutil.copy(p, PKG/f)
verdict = json.load(open(RESULTS/'e15_verdict.json'))
order = ['W1_full', 'W0_rule_only']
best = next((v for v in order if verdict[v]['status'] == 'RULE-SUCCESS'),
       next((v for v in order if verdict[v]['status'] == 'RULE-PARTIAL'), 'W1_full'))
shutil.copytree(RESULTS/'models'/best, PKG/f'model_{best}')
print('packaged best candidate:', best)
zip_path = shutil.make_archive('/content/geo_e15_results', 'zip', PKG)
print('packaged:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('manual download: use the Files sidebar ->', zip_path)

In [ ]:
# ── OPTIONAL: also copy results to your Google Drive ─────────────────────────
SAVE_TO_DRIVE = False   # flip to True, then run this cell
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    dest = '/content/drive/MyDrive/geo_e15_results.zip'
    shutil.copy('/content/geo_e15_results.zip', dest)
    print('saved to Drive:', dest)